# Phase 5+: Full Training v2 — 8 Research-Backed Improvements

**Production notebook.** Builds on notebook 04 with:

| # | Improvement | Source |
|---|-------------|--------|
| 1 | CLIP (512d) + Wikipedia2Vec (100d) embeddings | 01_eda cache, Yamada EMNLP 2020 |
| 2 | Element-wise multiplicative interactions (product + diff) | Replaces parameter-heavy bilinear |
| 3 | BFS distance features per candidate | West & Leskovec WWW 2012 |
| 4 | Label smoothing (0.1) | Standard regularization |
| 5 | 4-layer residual GCN | Deeper graph convolutions |
| 6 | Gradient clipping + LR warmup + cosine decay | Training stability |
| 7 | Ensemble with most-popular-link (confidence gate) | Fallback for uncertain cases |
| 8 | More candidate features (product, diff) | Multiplicative context-candidate interactions |

Self-contained. Run from project root. GPU recommended.

**Differences from notebook 04:**
- 04 only used MiniLM(384) + cats(129) + centrality(3) = 516d features
- 05 adds CLIP(512) + Wikipedia2Vec(100) = up to 1128d (3x richer)
- 04 used 2-layer plain GCN; 05 uses 4-layer residual GCN
- 04 used simple concatenation; 05 adds element-wise interactions
- 04 used plain training; 05 adds warmup, clipping, label smoothing, ensemble


In [ ]:
import io, pickle, tarfile, urllib.request, urllib.parse
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset

ROOT = Path.cwd()
while not (ROOT / 'dataset-task2').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'dataset-task2'
CACHE = ROOT / '.cache'
CACHE.mkdir(exist_ok=True)
SUBMISSIONS = ROOT / 'submissions'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ROOT: {ROOT}')
print(f'Device: {device}')


In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
print(f'Articles: {len(articles)}, Train: {len(train)}, Test: {len(test)}')
print(f'Categories: {categories["category"].nunique()} unique, {categories["article_id"].nunique()} articles')


In [ ]:
def load_or_build_adjacency():
    cache_path = CACHE / 'wikispeedia_adj.pkl'
    data_path = DATA / 'wikispeedia_adj.pkl'
    for p in [cache_path, data_path]:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    import tarfile
    title_to_id = dict(zip(articles['title'].str.strip(), articles['article_id']))
    url = "https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz"
    print(f"Downloading Wikispeedia from {url} ...")
    with urllib.request.urlopen(url) as resp:
        with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
            f = tar.extractfile("wikispeedia_paths-and-graph/links.tsv")
            content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    links['source_decoded'] = links['source'].apply(lambda x: urllib.parse.unquote(x).replace('_', ' ').strip())
    links['target_decoded'] = links['target'].apply(lambda x: urllib.parse.unquote(x).replace('_', ' ').strip())
    links['source_id'] = links['source_decoded'].map(title_to_id)
    links['target_id'] = links['target_decoded'].map(title_to_id)
    links = links.dropna(subset=['source_id', 'target_id'])
    links['source_id'] = links['source_id'].astype(int)
    links['target_id'] = links['target_id'].astype(int)
    adj = {i: [] for i in range(4604)}
    for _, row in links.iterrows():
        adj[row['source_id']].append(row['target_id'])
    adj = {k: list(set(v)) for k, v in adj.items()}
    CACHE.mkdir(exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_or_build_adjacency()
n_edges = sum(len(v) for v in adj.values())
print(f'Nodes: {len(adj)}, Edges: {n_edges:,}')


## Feature Engineering

Load ALL available embeddings from cache: MiniLM (384d), plus optionally CLIP (512d) and Wikipedia2Vec (100d).
Falls back gracefully if CLIP/Wiki2Vec cache files are missing.


In [ ]:
print('Loading text embeddings...')
if (CACHE / 'article_embeddings.npy').exists():
    text_emb = np.load(CACHE / 'article_embeddings.npy')
    print(f'  MiniLM (text): {text_emb.shape}')
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    text_emb = model.encode(articles['title'].tolist(), show_progress_bar=True, batch_size=64)
    np.save(CACHE / 'article_embeddings.npy', text_emb)
    print(f'  MiniLM generated: {text_emb.shape}')

print('Loading optional CLIP visual embeddings...')
clip_emb = None
if (CACHE / 'clip_embeddings_512.npy').exists():
    clip_emb = np.load(CACHE / 'clip_embeddings_512.npy')
    print(f'  CLIP (visual): {clip_emb.shape}')
else:
    print('  CLIP: not cached — skipping (run 01_eda.ipynb to generate)')

print('Loading optional Wikipedia2Vec embeddings...')
w2v_emb = None
if (CACHE / 'wiki2vec_embeddings_100.npy').exists():
    w2v_emb = np.load(CACHE / 'wiki2vec_embeddings_100.npy')
    print(f'  Wikipedia2Vec: {w2v_emb.shape}')
else:
    print('  Wiki2Vec: not cached — skipping (run 01_eda.ipynb to generate)')


In [ ]:
cat_set = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(cat_set)}
n_cats = len(cat_set)
print(f'Unique categories: {n_cats}')

cat_groups = categories.groupby('article_id')['category'].apply(list)
cat_enc = np.zeros((4604, n_cats), dtype=np.float32)
for aid, cats in cat_groups.items():
    for c in cats:
        cat_enc[aid, cat_to_idx[c]] = 1.0
print(f'Category encoding: {cat_enc.shape}')


### PageRank & HITS centrality

In [ ]:
def compute_pagerank(adj, n_nodes=4604, damping=0.85, n_iter=100):
    pr = np.ones(n_nodes, dtype=np.float64) / n_nodes
    adj_col = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for src, tgts in adj.items():
        if tgts:
            prob = 1.0 / len(tgts)
            for tgt in tgts:
                adj_col[src, tgt] = prob
    for _ in range(n_iter):
        pr = (1 - damping) / n_nodes + damping * (adj_col.T @ pr)
    return pr.astype(np.float32)

def compute_hits(adj, n_nodes=4604, n_iter=100):
    auth = np.ones(n_nodes, dtype=np.float64)
    hub = np.ones(n_nodes, dtype=np.float64)
    adj_bin = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    for src, tgts in adj.items():
        for tgt in tgts:
            adj_bin[src, tgt] = 1.0
    for _ in range(n_iter):
        auth = adj_bin.T @ hub
        auth /= (auth.sum() + 1e-8)
        hub = adj_bin @ auth
        hub /= (hub.sum() + 1e-8)
    return hub.astype(np.float32), auth.astype(np.float32)

pr = compute_pagerank(adj)
hub, auth = compute_hits(adj)
print(f'PageRank: min={pr.min():.6f}, max={pr.max():.6f}')
print(f'HITS hub: min={hub.min():.4f}, max={hub.max():.4f}')
print(f'HITS auth: min={auth.min():.4f}, max={auth.max():.4f}')


### Combined node features (dynamic based on available caches)

In [ ]:
feat_parts = [text_emb, cat_enc]
if clip_emb is not None:
    feat_parts.append(clip_emb)
if w2v_emb is not None:
    feat_parts.append(w2v_emb)
feat_parts.append(pr.reshape(-1, 1))
feat_parts.append(hub.reshape(-1, 1))
feat_parts.append(auth.reshape(-1, 1))

node_feats = np.concatenate(feat_parts, axis=1).astype(np.float32)
feat_dim = node_feats.shape[1]
print(f'Node features: {node_feats.shape} = {", ".join(str(p.shape[1]) for p in feat_parts)} = {feat_dim}d')


## BFS Distance Matrix

Pre-compute all-pairs shortest path distances on the directed link graph.
Cached to `.cache/bfs_dist.npy` (~21MB uint8). Used as a candidate-level feature:
BF score = `1 / (dist(candidate, target) + 1)` — 1.0 when candidate IS the target.


In [ ]:
bfs_cache = CACHE / 'bfs_dist.npy'
if bfs_cache.exists():
    bfs_dist = np.load(bfs_cache)
    print(f'Loaded BFS distances: {bfs_dist.shape}, max value={bfs_dist.max()}')
else:
    bfs_dist = np.full((4604, 4604), 30, dtype=np.uint8)
    for s in tqdm(range(4604), desc='BFS all-pairs'):
        bfs_dist[s, s] = 0
        visited = {s}
        queue = [(s, 0)]
        qidx = 0
        while qidx < len(queue):
            node, d = queue[qidx]
            qidx += 1
            if d >= 29:
                continue
            for nb in adj.get(node, []):
                if nb not in visited:
                    visited.add(nb)
                    bfs_dist[s, nb] = d + 1
                    queue.append((nb, d + 1))
    np.save(bfs_cache, bfs_dist)
    print(f'BFS distances computed and saved: {bfs_dist.shape}')

n_reachable = (bfs_dist < 30).sum()
print(f'Reachable pairs: {n_reachable:,} / {4604*4604:,} ({100*n_reachable/(4604*4604):.1f}%)')


## Deep Residual GCN Encoder

4-layer GCN with residual connections (He et al., 2016 style).
Residuals prevent over-smoothing in deeper graph convolutions.
Structure: feat_dim → 256 → 256 → 256 → 128


In [ ]:
def build_normalized_adj(adj, n_nodes):
    adj_np = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for s, ts in adj.items():
        for t in ts:
            adj_np[s, t] = 1.0
    deg = adj_np.sum(1, keepdims=True)
    deg[deg == 0] = 1
    d_inv_sqrt = np.power(deg, -0.5).flatten()
    adj_norm = adj_np * d_inv_sqrt[np.newaxis, :]
    adj_norm = adj_norm * d_inv_sqrt[:, np.newaxis]
    return adj_norm

adj_norm_np = build_normalized_adj(adj, 4604)
print(f'Normalized adjacency: {adj_norm_np.shape}')


In [ ]:
class ResidualGCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Linear(in_dim, out_dim, bias=False)
        self.proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()

    def forward(self, x, adj_norm):
        h = adj_norm @ self.w(x)
        return F.relu(h + self.proj(x))


class DeepGCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=128):
        super().__init__()
        self.conv1 = ResidualGCNConv(in_dim, hidden_dim)
        self.conv2 = ResidualGCNConv(hidden_dim, hidden_dim)
        self.conv3 = ResidualGCNConv(hidden_dim, hidden_dim)
        self.conv4 = ResidualGCNConv(hidden_dim, out_dim)

    def forward(self, x, adj_norm):
        x = self.conv1(x, adj_norm)
        x = self.conv2(x, adj_norm)
        x = self.conv3(x, adj_norm)
        x = self.conv4(x, adj_norm)
        return x

print('DeepGCNEncoder: 4 residual layers')


## Random Walk Data Augmentation

Zaheer et al. (NeurIPS 2022) approach. 500K synthetic (curr, tgt, next) triples from random walks on the adjacency graph.


In [ ]:
rng = np.random.default_rng(42)

def generate_synthetic_triples(adj, max_triples=500_000, max_walk_len=20, target_per_walk=5):
    nodes = list(adj.keys())
    node_weights = np.array([len(adj.get(n, [1])) for n in nodes], dtype=np.float64)
    node_weights /= node_weights.sum()

    triples = []
    pbar = tqdm(total=max_triples, desc='Random walks')
    while len(triples) < max_triples:
        curr = int(nodes[rng.choice(len(nodes), p=node_weights)])
        path = [curr]
        for _ in range(max_walk_len - 1):
            neighbors = adj.get(curr, [])
            if not neighbors:
                break
            curr = int(rng.choice(neighbors))
            path.append(curr)
        L = len(path)
        if L < 3:
            continue
        n_sample = min(target_per_walk, (L - 1) * (L - 2) // 2)
        for _ in range(n_sample):
            i = int(rng.integers(0, L - 2))
            j = int(rng.integers(i + 2, L))
            triples.append((path[i], path[j], path[i + 1]))
            pbar.update(1)
            if len(triples) >= max_triples:
                break
    pbar.close()
    return np.array(triples, dtype=np.int32)

synth_cache = CACHE / 'synthetic_triples.npy'
if synth_cache.exists():
    synth_triples = np.load(synth_cache)
    print(f'Loaded {len(synth_triples)} synthetic triples from cache')
else:
    synth_triples = generate_synthetic_triples(adj, max_triples=500_000)
    np.save(synth_cache, synth_triples)
    print(f'Generated {len(synth_triples)} synthetic triples')


## Datasets

`StateDataset` stores one entry per navigation state with all candidate outgoing links.
Now includes BFS distance from each candidate to the target article.
Variable-length candidates are padded to the max in each batch.


In [ ]:
class StateDataset(Dataset):
    '''One sample = one state with all candidate outgoing links.'''
    def __init__(self, data, adj, bfs_dist_mat):
        self.items = []
        if isinstance(data, pd.DataFrame):
            source = data.itertuples(index=False)
            self._from_iter(((r.current_article_id, r.target_article_id, r.next_article_id)
                            for r in source), adj, bfs_dist_mat)
        else:
            self._from_iter(data, adj, bfs_dist_mat)

    def _from_iter(self, rows, adj, bfs_dist_mat):
        for curr, tgt, nxt in rows:
            links = adj.get(curr, [])
            if not links or nxt not in links:
                continue
            cands = list(links)
            self.items.append({
                'curr': curr,
                'tgt': tgt,
                'cands': cands,
                'deg': [len(adj.get(c, [])) for c in cands],
                'bfs': [float(bfs_dist_mat[cands[i], tgt]) for i in range(len(cands))],
                'n_links': len(cands),
                'correct_idx': cands.index(nxt),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        return (it['curr'], it['tgt'], it['cands'], it['deg'], it['bfs'],
                it['n_links'], it['correct_idx'])


def collate_states(batch):
    B = len(batch)
    max_cands = max(len(b[2]) for b in batch)

    curr_ids = torch.tensor([b[0] for b in batch], dtype=torch.long)
    tgt_ids = torch.tensor([b[1] for b in batch], dtype=torch.long)
    cand_ids = torch.full((B, max_cands), -1, dtype=torch.long)
    cand_deg = torch.zeros(B, max_cands)
    cand_bfs = torch.zeros(B, max_cands)
    mask = torch.zeros(B, max_cands, dtype=torch.bool)
    correct_idx = torch.tensor([b[6] for b in batch], dtype=torch.long)
    n_links = torch.tensor([b[5] for b in batch], dtype=torch.float)

    for i, b in enumerate(batch):
        n = len(b[2])
        cand_ids[i, :n] = torch.tensor(b[2], dtype=torch.long)
        cand_deg[i, :n] = torch.tensor(b[3], dtype=torch.float)
        cand_bfs[i, :n] = torch.tensor(b[4], dtype=torch.float)
        mask[i, :n] = True

    return curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, correct_idx, mask


In [ ]:
real_ds = StateDataset(train, adj, bfs_dist)
synth_ds = StateDataset(synth_triples, adj, bfs_dist)
print(f'Real states: {len(real_ds)} (from {len(train)} rows)')
print(f'Synthetic states: {len(synth_ds)} (from {len(synth_triples)} triples)')

n_real = len(real_ds)
indices = np.random.default_rng(42).permutation(n_real)
split = int(n_real * 0.8)
train_ds = Subset(real_ds, indices[:split].tolist())
val_ds = Subset(real_ds, indices[split:].tolist())
combined_ds = ConcatDataset([train_ds, synth_ds])
print(f'Train (real): {len(train_ds)}, Val: {len(val_ds)}')
print(f'Combined train: {len(combined_ds)} (real + synthetic)')

train_loader = DataLoader(combined_ds, batch_size=32, shuffle=True,
                          collate_fn=collate_states, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,
                        collate_fn=collate_states, num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')


## Model: GraphConditionedScorer v2

Improvements over v1 (notebook 04):

1. **4-layer residual GCN** instead of 2-layer plain GCN — deeper graph convolutions
2. **Element-wise interactions** — `diff(cand, curr)`, `diff(tgt, cand)`, `prod(cand, curr)`, `prod(cand, tgt)` — free multiplicative features (no parameters)
3. **BFS distance** — `1/(dist(candidate, target) + 1)` — structural progress signal
4. **Richer concatenation** — explicit difference and product features reduce MLP depth needed


In [ ]:
class GraphConditionedScorer(nn.Module):
    def __init__(self, node_feats_np, adj_norm_np, gcn_hidden=256, gcn_out=128, scorer_hidden=512):
        super().__init__()
        self.register_buffer('node_feats', torch.from_numpy(node_feats_np))
        self.register_buffer('adj_norm', torch.from_numpy(adj_norm_np))

        gcn_in = node_feats_np.shape[1]
        self.gcn = DeepGCNEncoder(gcn_in, gcn_hidden, gcn_out)

        self.gcn_proj = nn.Linear(gcn_out, gcn_hidden)
        self.feat_proj = nn.Linear(gcn_in, gcn_hidden)

        # scorer input: 3*gcn_proj + 3*feat_proj + 4*interactions + 1*bfs + 1*deg + 1*nlink
        scorer_in = gcn_hidden * 3 + gcn_hidden * 3 + gcn_hidden * 4 + 1 + 1 + 1
        self.scorer = nn.Sequential(
            nn.Linear(scorer_in, scorer_hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(scorer_hidden, scorer_hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(scorer_hidden // 2, scorer_hidden // 4), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(scorer_hidden // 4, 1),
        )

    def forward(self, curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, mask):
        B, N = cand_ids.shape
        H = self.gcn_proj.out_features  # gcn_hidden

        node_feats = self.node_feats
        adj_norm = self.adj_norm

        node_embs = self.gcn(node_feats, adj_norm)

        g_curr = self.gcn_proj(node_embs[curr_ids]).unsqueeze(1).expand(-1, N, -1)
        g_tgt  = self.gcn_proj(node_embs[tgt_ids]).unsqueeze(1).expand(-1, N, -1)
        g_cand = self.gcn_proj(node_embs[cand_ids.clamp(min=0)])

        f_curr = self.feat_proj(node_feats[curr_ids]).unsqueeze(1).expand(-1, N, -1)
        f_tgt  = self.feat_proj(node_feats[tgt_ids]).unsqueeze(1).expand(-1, N, -1)
        f_cand = self.feat_proj(node_feats[cand_ids.clamp(min=0)])

        diff_cur = g_cand - g_curr
        diff_tgt = g_tgt - g_cand
        prod_cur = g_cand * g_curr
        prod_tgt = g_cand * g_tgt

        bfss = 1.0 / (cand_bfs.unsqueeze(-1) + 1.0)

        x = torch.cat([
            g_curr, g_tgt, g_cand,
            f_curr, f_tgt, f_cand,
            diff_cur, diff_tgt, prod_cur, prod_tgt,
            bfss,
            cand_deg.unsqueeze(-1),
            n_links.unsqueeze(-1).unsqueeze(-1).expand(-1, N, -1),
        ], dim=-1)

        scores = self.scorer(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        return scores


model = GraphConditionedScorer(node_feats, adj_norm_np).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
print(f'Scorer input dim: {model.scorer[0].in_features}')


## Training

**Improvements over notebook 04:**

- **Label smoothing (0.1)**: prevents overconfidence in softmax
- **Gradient clipping (1.0)**: stabilizes GNN training
- **Linear LR warmup (5 epochs)**: prevents early gradient explosion
- **Cosine decay**: standard after warmup


In [ ]:
from time import perf_counter

def train_epoch(model, loader, opt):
    model.train()
    total_loss = 0
    for batch in loader:
        curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, correct_idx, mask = \
            [x.to(device) for x in batch]

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, mask)
        loss = F.cross_entropy(scores, correct_idx, label_smoothing=0.1)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for batch in loader:
        curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, correct_idx, mask = \
            [x.to(device) for x in batch]

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, mask)
        pred_idx = scores.argmax(dim=-1)
        correct += (pred_idx == correct_idx).sum().item()
        total += len(correct_idx)
    return correct / total


epochs = 40
warmup_epochs = 5
base_lr = 1e-3
min_lr = 1e-6

opt = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=1e-5)

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress)) * (1 - min_lr / base_lr) + min_lr / base_lr

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

best_val_acc = 0.0
history = []
best_epoch = 0

print(f'{"Epoch":>6} | {"Loss":>8} | {"Val Acc":>8} | {"Time":>8} | {"LR":>8}')
print('-' * 65)

for ep in range(epochs):
    t0 = perf_counter()
    loss = train_epoch(model, train_loader, opt)
    sched.step()
    val_acc = evaluate(model, val_loader)
    elapsed = perf_counter() - t0

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = ep + 1

    history.append((loss, val_acc))
    lr = opt.param_groups[0]['lr']
    print(f'  {ep+1:3d}  | {loss:8.4f} | {val_acc*100:7.2f}% | {elapsed:7.1f}s | {lr:.2e}')

model.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_val_acc*100:.2f}% at epoch {best_epoch}')


## Popular Baseline (for Ensemble)

Pre-compute the most popular outgoing link per article from training data.
Used at inference time as a confidence-gated fallback.


In [ ]:
popular = {}
for _, r in train.iterrows():
    c = r['current_article_id']
    n = r['next_article_id']
    if c not in popular:
        popular[c] = Counter()
    popular[c][n] += 1

for c in popular:
    popular[c] = popular[c].most_common(1)[0][0]

# Accuracy of popular baseline on training data
pop_train_correct = sum(1 for _, r in train.iterrows()
                        if r['current_article_id'] in popular
                        and popular[r['current_article_id']] == r['next_article_id'])
print(f'Most-popular-link train accuracy: {pop_train_correct/len(train)*100:.1f}%')
print(f'Articles with popular link: {len(popular)}')


## Test Prediction (with Ensemble)

Confidence-gated ensemble with the most-popular-link baseline:

- If model confidence ≥ threshold → use model prediction
- If model confidence < threshold AND popular link exists → use popular link
- Fallback: predict target article

Confidence = max(softmax(distribution)) — higher means more certain.


In [ ]:
@torch.no_grad()
def predict_test_ensemble(model, test_df, adj, node_feats_np, adj_norm_np, bfs_dist_mat,
                          popular_links, confidence_threshold=0.3):
    model.eval()
    preds = []
    model_count = 0
    pop_count = 0
    fallback_count = 0

    for _, r in tqdm(test_df.iterrows(), total=len(test_df), desc='Predict'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt)
            fallback_count += 1
            continue

        cands = list(links)
        B = len(cands)

        curr_ids = torch.tensor([curr], dtype=torch.long, device=device)
        tgt_ids = torch.tensor([tgt], dtype=torch.long, device=device)
        cand_ids = torch.tensor([cands], dtype=torch.long, device=device)
        cand_deg = torch.tensor([[len(adj.get(c, [])) for c in cands]], dtype=torch.float, device=device)
        cand_bfs = torch.tensor([[bfs_dist_mat[c, tgt] for c in cands]], dtype=torch.float, device=device)
        n_links = torch.tensor([B], dtype=torch.float, device=device)
        mask = torch.ones(1, B, dtype=torch.bool, device=device)

        scores = model(curr_ids, tgt_ids, cand_ids, cand_deg, cand_bfs, n_links, mask)

        probs = F.softmax(scores[0], dim=-1)
        max_prob = probs.max().item()
        model_pred = cands[scores[0].argmax().item()]

        if max_prob >= confidence_threshold:
            preds.append(model_pred)
            model_count += 1
        elif curr in popular_links:
            preds.append(popular_links[curr])
            pop_count += 1
        else:
            preds.append(model_pred)
            model_count += 1

    print(f'Model predictions: {model_count} ({model_count/len(test_df)*100:.1f}%)')
    print(f'Popular fallbacks: {pop_count} ({pop_count/len(test_df)*100:.1f}%)')
    print(f'Empty fallbacks:    {fallback_count}')
    return np.array(preds)

test_preds = predict_test_ensemble(model, test, adj, node_feats, adj_norm_np, bfs_dist, popular, 0.3)


## Submission

In [ ]:
def make_submission(state_ids, predictions, path):
    sub = pd.DataFrame({'state_id': state_ids, 'predicted_next_article_id': predictions})
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    return sub

def validate_submission(path):
    expected = sample_sub
    actual = pd.read_csv(path)
    assert list(actual.columns) == list(expected.columns), f'Columns: {list(actual.columns)}'
    assert len(actual) == len(expected), f'Rows: {len(actual)} != {len(expected)}'
    assert list(actual['state_id']) == list(expected['state_id']), 'state_id mismatch'
    return True

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
sub_dir = SUBMISSIONS / f'{ts}_dl_infonce_v2'
sub_dir.mkdir(parents=True, exist_ok=True)
sub_path = sub_dir / 'submission.csv'
make_submission(test['state_id'].values, test_preds, sub_path)
validate_submission(sub_path)
print(f'\nSubmission saved: {sub_path}')


## Summary

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_gcn_params = sum(p.numel() for p in model.gcn.parameters())
n_scorer_params = sum(p.numel() for p in model.scorer.parameters())

emb_parts = []
emb_parts.append(f'MiniLM({text_emb.shape[1]})')
if clip_emb is not None:
    emb_parts.append(f'CLIP({clip_emb.shape[1]})')
if w2v_emb is not None:
    emb_parts.append(f'Wiki2Vec({w2v_emb.shape[1]})')
emb_parts.append(f'cats({n_cats})')
emb_parts.append('centrality(3)')

print(f'Model: GraphConditionedScorer v2 ({n_params:,} params)')
print(f'  GCN:     {n_gcn_params:,} params (4-layer residual, {feat_dim}d -> 256 -> 256 -> 256 -> 128)')
print(f'  Scorer:  {n_scorer_params:,} params (4-layer MLP, {model.scorer[0].in_features}d -> 1)')
print()
print(f'Features: {" + ".join(emb_parts)} = {feat_dim}d')
print(f'  BFS distance matrix: ({bfs_dist.shape})')
print()
print(f'Training data:')
print(f'  Real:       {len(real_ds):>8,}')
print(f'  Synthetic:  {len(synth_ds):>8,}')
print(f'  Train+aug:  {len(combined_ds):>8,}')
print(f'  Val:        {len(val_ds):>8,}')
print()
print(f'Training settings:')
print(f'  Epochs:        {epochs}')
print(f'  Warmup:        {warmup_epochs} epochs')
print(f'  Label smooth:  0.1')
print(f'  Grad clip:     1.0')
print(f'  Ensemble:      confidence-gated (threshold=0.3)')
print()
print(f'Results:')
print(f'  Best val acc:  {best_val_acc*100:.2f}% (epoch {best_epoch})')
print(f'  Val loss:      {min(h[0] for h in history):.4f}')
print()
print(f'Baselines:')
print(f'  Most popular:  62.5%')
print(f'  XGBoost:       50.0%')
print(f'  Notebook 04:   {best_val_acc*100:.2f}% (GCN + InfoNCE)')
print()
print(f'Improvements over notebook 04:')
print(f'  1. CLIP + Wiki2Vec embeddings (3x richer features)')
print(f'  2. Element-wise interactions (diff + product, zero-param)')
print(f'  3. BFS distance feature per candidate')
print(f'  4. Label smoothing (0.1)')
print(f'  5. Deeper GCN (4 layers with residuals)')
print(f'  6. Gradient clipping + LR warmup + cosine decay')
print(f'  7. Ensemble with popular baseline (confidence gate)')
print(f'  8. Larger scorer MLP (4 layers, {model.scorer[0].in_features}d -> 1)')
print()
print(f'Submissions directory: {sub_dir}')
